# Text Generation with RNNs — NumPy vs TensorFlow vs PyTorch

This notebook trains a stacked (multi-layer) recurrent neural network three
ways and compares them on the **same data, architecture, and hyper-parameters**:

1. **Manual (NumPy)** — implemented from scratch in
   [`rnn_scratch_multi_layer.py`](rnn_scratch_multi_layer.py).
2. **TensorFlow / Keras** — [`rnn_tensorflow.py`](rnn_tensorflow.py).
3. **PyTorch** — [`rnn_pytorch.py`](rnn_pytorch.py).

All three expose the same interface (`train()`, `predict()`), so the shared
helpers in [`utils.py`](utils.py) — `evaluate`, `generate` — and
[`compare.py`](compare.py)'s `compare_models` work on every model unchanged.

**Pipeline:** raw text → vocabulary → training sequences → train/test split →
train the three models → compare train/test accuracy → generate text.

| Section | Contents |
|---------|----------|
| 1 | Imports |
| 2 | Data |
| 3 | Character level — 3-way comparison |
| 4 | Word level — Word2Vec · GloVe · FastText · BERT (each a 3-way comparison) |

> **Note on the comparison.** The frameworks train with **Adam** while the
> from-scratch model uses plain **SGD + gradient clipping**, so their learned
> weights — and accuracies — differ. This is a realistic "library vs scratch"
> comparison, not a bit-for-bit match.

## 1. Imports

In [1]:
import importlib

import numpy as np

import rnn_scratch_multi_layer, rnn_tensorflow, rnn_pytorch, utils, compare

# Reload local modules so edits to the .py files are picked up without
# having to restart the kernel.
for _m in (rnn_scratch_multi_layer, rnn_tensorflow, rnn_pytorch, utils, compare):
    importlib.reload(_m)

from rnn_scratch_multi_layer import RNN
from rnn_tensorflow import KerasRNN
from rnn_pytorch import TorchRNN
from utils import generate_dataset, train_test_split, evaluate, predict_next, generate
from compare import compare_models

## 2. Data

A small block of text to learn from. With a corpus this size the model can only
learn surface statistics, but it is enough to demonstrate and compare the RNNs.

In [2]:
sentences = [
'Machine Learning (ML) is a branch of Artificial Intelligence (AI)',
'that enables computers to learn from data and improve their performance',
'without being explicitly programmed. ML algorithms identify patterns in',
'historical data and use those patterns to make predictions or decisions on new data.',
'Machine learning is widely used in various applications such as recommendation systems,',
'image recognition, speech processing, fraud detection, healthcare diagnostics,',
'and autonomous vehicles. The main types of machine learning are supervised learning,',
'unsupervised learning, and reinforcement learning. As the amount of available data continues to grow,',
'machine learning plays an increasingly important role in helping organizations automate tasks,',
'gain insights, and make data-driven decisions.'
]
sentence_words = [s.split() for s in sentences]
words = [w for s in sentence_words for w in s]
text = " ".join(words)

In [3]:
def train_models(X_train, Y_train, hidden_layers, lr, iters, task):

    print('-'*100)
    print('Traning Manual RNN')
    
    manual_rnn = RNN(X_train, Y_train, hidden_layers=hidden_layers,
                 learning_rate=lr, iterations=iters, task=task)
    manual_rnn.train()
    print()
    print('-'*100)
    print('Traning Tensorflow RNN')
    keras_rnn = KerasRNN(X_train, Y_train, hidden_layers=hidden_layers,
                        learning_rate=lr, iterations=iters, task= task).train()
    print()
    print('-'*100)
    print('Traning PyTorch RNN')
    torch_rnn = TorchRNN(X_train, Y_train, hidden_layers=hidden_layers,
                        learning_rate=lr, iterations=iters, task=task).train()

    print()
    print('-'*100)
    
    tranined_models = {"manual (numpy)": manual_rnn,
                "tensorflow": keras_rnn,
                "pytorch": torch_rnn}
    
    return tranined_models

## 3. Character-level Text Generation

Slide a window of length `T_x` over the text: each input `X` is a chunk of
characters and the target `Y` is that chunk shifted one character left, so the
model learns to predict the *next* character at every step. Arrays are one-hot
encoded into `(vocab_size, m, T_x)`.

In [4]:
# One-hot character sequences (word_vectors is unused in char mode).
char_X, char_Y, char_to_index, index_to_char = generate_dataset(
    text, T_x=10, is_char=True, word_vectors=[], seq_length=25
)
print(f"X: {char_X.shape}   Y: {char_Y.shape}   (vocab_size, m, T_x)")

# Hold out 20% of the sequences for testing.
X_train, X_test, Y_train, Y_test = train_test_split(char_X, char_Y, test_size=0.2)

Generated 754 training sequences of length 10 from a corpus of 790 tokens, with a vocabulary of 35 unique characters.
X: (35, 754, 10)   Y: (35, 754, 10)   (vocab_size, m, T_x)
Split 754 sequences -> 603 train / 151 test.


### 3.1 Train the three models

Same `hidden_layers` and hyper-parameters for all three, trained on the same
training split.

In [5]:
hidden_layers, lr, iters, task = (50,), 0.001, 1000, 'classification'
print('Character Models')
char_models = train_models(
    X_train, Y_train, hidden_layers,lr, iters, task
)

Character Models
----------------------------------------------------------------------------------------------------
Traning Manual RNN
Iteration: 0, Loss: 35.55347296301148
Iteration: 100, Loss: 24.193095931865383
Iteration: 200, Loss: 15.941827371116998
Iteration: 300, Loss: 11.185431747551348
Iteration: 400, Loss: 8.612625150822923
Iteration: 500, Loss: 7.202985087653179
Iteration: 600, Loss: 6.2478300322292055
Iteration: 700, Loss: 5.576449736282579
Iteration: 800, Loss: 5.221726110944431
Iteration: 900, Loss: 4.93027450523235

----------------------------------------------------------------------------------------------------
Traning Tensorflow RNN
[Keras] trained 1000 epochs, final loss 0.6342

----------------------------------------------------------------------------------------------------
Traning PyTorch RNN
[PyTorch] iter 0, loss 3.5859
[PyTorch] iter 100, loss 2.8612
[PyTorch] iter 200, loss 2.3151
[PyTorch] iter 300, loss 1.7934
[PyTorch] iter 400, loss 1.3905
[PyTorch] 

### 3.2 Compare

`compare_models` scores each trained model on the train/test splits and
generates a sequence from the same seed character.

In [6]:
compare_models(
    char_models, X_train, Y_train, X_test, Y_test,
    embedding=char_to_index, decoder=index_to_char,
    seed_word='M', is_char=True, num_gen=100,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.8486   0.7430
  tensorflow         0.8154   0.6079
  pytorch            0.8174   0.6053

=== generation from 'M' ===
  manual (numpy)   ML algorithms identify patterns in historical data and improve their performance without being explic
  tensorflow       ML algorithos inatlons auch prosestion, sutemsin bys co makan is dea contiorn macan important rocesti
  pytorch          Machine learning (ML) is a branc lecicily pattere thes tten deasithoupe Lealathoreathecrans, amake pr


## 4. Word-level Prediction

Now each input token is a dense **word embedding** instead of a one-hot
character; targets stay one-hot over the vocabulary, so the model still runs in
`task="classification"` and predicts the next word.

Each subsection feeds the same corpus through a different embedding
(Word2Vec · GloVe · FastText · BERT) and runs the **full 3-way comparison**
(manual / TensorFlow / PyTorch).

### 4.1 Word2Vec embeddings

Train a Word2Vec model on our sentences (skip-gram) and use its vectors as the RNN's input features. This section also demonstrates a **2-layer stack** (`hidden_layers=(100, 64)`).

In [7]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
word_vectors = w2v_model.wv

In [8]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=word_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (vector_size, m, T_x)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (100, 102, 5)   Y: (88, 102, 5)   (vector_size, m, T_x)
Split 102 sequences -> 82 train / 20 test.


In [9]:
hidden_layers, lr, iters, task = (100, 64), 0.01, 1000, 'classification'

print('Word2Vec Embedding')
w2v_models = train_models(
    X_train, Y_train, hidden_layers,lr, iters, task
)

Word2Vec Embedding
----------------------------------------------------------------------------------------------------
Traning Manual RNN
Iteration: 0, Loss: 22.38668390747042
Iteration: 100, Loss: 21.464560733641097
Iteration: 200, Loss: 21.027259602578734
Iteration: 300, Loss: 20.925004130129786
Iteration: 400, Loss: 20.90713526930562
Iteration: 500, Loss: 20.926157655262852
Iteration: 600, Loss: 20.781438416542215
Iteration: 700, Loss: 17.87269433008615
Iteration: 800, Loss: 15.1160042556975
Iteration: 900, Loss: 10.785435606210621

----------------------------------------------------------------------------------------------------
Traning Tensorflow RNN
[Keras] trained 1000 epochs, final loss 0.0535

----------------------------------------------------------------------------------------------------
Traning PyTorch RNN
[PyTorch] iter 0, loss 4.4858
[PyTorch] iter 100, loss 0.1054
[PyTorch] iter 200, loss 0.0590
[PyTorch] iter 300, loss 0.0558
[PyTorch] iter 400, loss 0.0547
[PyTor

In [10]:
compare_models(
    w2v_models, X_train, Y_train, X_test, Y_test,
    embedding=word_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.6805   0.0300
  tensorflow         0.9659   0.2600
  pytorch            0.9659   0.4100

=== generation from 'Machine' ===
  manual (numpy)   Machine (ML) speech in historical data learning, important role learning fraud
  tensorflow       Machine Learning (ML) is a branch of Artificial Intelligence new patterns
  pytorch          Machine learning is widely used in various applications such as recommendation


### 4.2 GloVe — pre-trained embeddings (Google-News word2vec)

Swap our small in-house vectors for Google's pre-trained 300-d word2vec vectors (trained on ~100B words of Google News).

In [11]:
# pre-trained vectors from the Google-News dataset (300d, 3M words)
import gensim.downloader as api
glove = api.load("word2vec-google-news-300")

In [12]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=glove)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (vector_size, m, T_x)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 72 training sequences of length 5 from a corpus of 78 tokens, with a vocabulary of 67 unique words.
X: (300, 72, 5)   Y: (67, 72, 5)   (vector_size, m, T_x)
Split 72 sequences -> 58 train / 14 test.


In [13]:
hidden_layers, lr, iters, task = (100,), 0.001, 1000,'classification'

print('Glove Embedding')
glove_models = train_models(
    X_train, Y_train, hidden_layers,lr, iters, task
)


Glove Embedding
----------------------------------------------------------------------------------------------------
Traning Manual RNN
Iteration: 0, Loss: 21.023691403644207
Iteration: 100, Loss: 18.918538253810013
Iteration: 200, Loss: 8.466742672510856
Iteration: 300, Loss: 2.5673820514665295
Iteration: 400, Loss: 1.2776961939963334
Iteration: 500, Loss: 0.8437959261994485
Iteration: 600, Loss: 0.6313322207139413
Iteration: 700, Loss: 0.5099760310578946
Iteration: 800, Loss: 0.4344486301098173
Iteration: 900, Loss: 0.3839499138905617

----------------------------------------------------------------------------------------------------
Traning Tensorflow RNN
[Keras] trained 1000 epochs, final loss 0.0371

----------------------------------------------------------------------------------------------------
Traning PyTorch RNN
[PyTorch] iter 0, loss 4.2052
[PyTorch] iter 100, loss 0.2803
[PyTorch] iter 200, loss 0.0885
[PyTorch] iter 300, loss 0.0564
[PyTorch] iter 400, loss 0.0467
[PyTo

In [14]:
compare_models(
    glove_models, X_train, Y_train, X_test, Y_test,
    embedding=glove, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.9759   0.9286
  tensorflow         0.9759   0.8000
  pytorch            0.9759   0.9286

=== generation from 'Machine' ===
  manual (numpy)   Machine learning is widely used in various applications such as recommendation
  tensorflow       Machine Learning is branch Artificial Intelligence that enables computers learn from
  pytorch          Machine learning is widely used in various applications such as recommendation


### 4.3 FastText embeddings

FastText builds word vectors from character n-grams, so it can embed even rare or out-of-vocabulary words. Trained here on our own sentences.

In [15]:
from gensim.models import FastText

ft_model = FastText(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
fasttext_vectors = ft_model.wv

In [16]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=fasttext_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (vector_size, m, T_x)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (100, 102, 5)   Y: (88, 102, 5)   (vector_size, m, T_x)
Split 102 sequences -> 82 train / 20 test.


In [17]:
hidden_layers, lr, iters, task = (50,), 0.01193, 1000,'classification'

print('FastText Embedding')
fasttext_models = train_models(
    X_train, Y_train, hidden_layers,lr, iters, task
)

FastText Embedding
----------------------------------------------------------------------------------------------------
Traning Manual RNN
Iteration: 0, Loss: 22.386681008108006
Iteration: 100, Loss: 21.588967898907907
Iteration: 200, Loss: 20.9882516944579
Iteration: 300, Loss: 20.925482056726473
Iteration: 400, Loss: 20.911574389310683
Iteration: 500, Loss: 20.905165081745025
Iteration: 600, Loss: 20.905210956614905
Iteration: 700, Loss: 20.893031818423342
Iteration: 800, Loss: 20.8958843742842
Iteration: 900, Loss: 20.893905782403905

----------------------------------------------------------------------------------------------------
Traning Tensorflow RNN
[Keras] trained 1000 epochs, final loss 0.0676

----------------------------------------------------------------------------------------------------
Traning PyTorch RNN
[PyTorch] iter 0, loss 4.4848
[PyTorch] iter 100, loss 1.7819
[PyTorch] iter 200, loss 0.7609
[PyTorch] iter 300, loss 0.4894
[PyTorch] iter 400, loss 0.3487
[PyTo

In [18]:
compare_models(
    fasttext_models, X_train, Y_train, X_test, Y_test,
    embedding=fasttext_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.0463   0.0100
  tensorflow         0.9659   0.0400
  pytorch            0.9659   0.0600

=== generation from 'Machine' ===
  manual (numpy)   Machine as such grow, recognition, Artificial performance historical role learning, types
  tensorflow       Machine Learning (ML) is a branch of various applications such as
  pytorch          Machine learning is widely used in in of such as their


### 4.4 BERT embeddings

Use a pre-trained BERT model to produce a contextual `[CLS]` embedding per word, then feed those 768-d vectors to the RNNs.

In [19]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

bert_word_to_vec = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = bert_model(**inputs)
    bert_word_to_vec[word] = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=bert_word_to_vec)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (vector_size, m, T_x)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (768, 102, 5)   Y: (88, 102, 5)   (vector_size, m, T_x)
Split 102 sequences -> 82 train / 20 test.


In [21]:
hidden_layers, lr, iters, task = (50,), 0.01193,1000,'classification'

print('Bert Embedding')
bert_models = train_models(
    X_train, Y_train, hidden_layers,lr, iters, task
)

Bert Embedding
----------------------------------------------------------------------------------------------------
Traning Manual RNN
Iteration: 0, Loss: 22.390091115049906
Iteration: 100, Loss: 5.181463449690178
Iteration: 200, Loss: 0.4861491021576814
Iteration: 300, Loss: 0.42481744772680907
Iteration: 400, Loss: 0.4044826305814128
Iteration: 500, Loss: 0.3966391724661092
Iteration: 600, Loss: 0.39115196765799276
Iteration: 700, Loss: 0.38815231365914343
Iteration: 800, Loss: 0.38666771999275995
Iteration: 900, Loss: 0.38435902317104353

----------------------------------------------------------------------------------------------------
Traning Tensorflow RNN
[Keras] trained 1000 epochs, final loss 0.0747

----------------------------------------------------------------------------------------------------
Traning PyTorch RNN
[PyTorch] iter 0, loss 4.4952
[PyTorch] iter 100, loss 0.0801
[PyTorch] iter 200, loss 0.0768
[PyTorch] iter 300, loss 0.0758
[PyTorch] iter 400, loss 0.0753
[

In [22]:
compare_models(
    bert_models, X_train, Y_train, X_test, Y_test,
    embedding=bert_word_to_vec, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.9561   0.7800
  tensorflow         0.9561   0.9400
  pytorch            0.9561   0.9400

=== generation from 'Machine' ===
  manual (numpy)   Machine learning plays an increasingly important role in helping organizations automate
  tensorflow       Machine learning plays an increasingly important role in helping organizations automate
  pytorch          Machine learning is widely used in various applications such as recommendation
